# ComfyUI en Google Colab con Google Drive
Este cuaderno permite instalar, configurar y ejecutar ComfyUI en Google Colab guardando modelos, nodos y salidas en Google Drive para total persistencia.

### ⚠️ Advertencias de Coste y Uso
- **Almacenamiento:** Los modelos pesados (SDXL, FLUX) ocupan entre 2 GB y 12 GB cada uno. Verifica el espacio disponible en tu Google Drive.
- **Límites de Colab:** Las sesiones gratuitas de GPU T4 son limitadas. Cierra el entorno de ejecución al terminar para ahorrar tiempo de GPU.

In [ ]:
# Paso 1: Verificación de GPU y Montaje de Google Drive
import os
import torch
from google.colab import drive

print("Verificando disponibilidad de GPU...")
if not torch.cuda.is_available():
    print("❌ ALERTA: No hay GPU activa. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución y selecciona GPU T4.")
else:
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}")

print("\nMontando Google Drive...")
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/ComfyUI'
print(f"Ruta base de ComfyUI establecida en: {BASE_PATH}")

In [ ]:
# Paso 2: Crear o verificar estructura de carpetas persistentes
import os

BASE_PATH = '/content/drive/MyDrive/ComfyUI'
directories = [
    f"{BASE_PATH}/models/checkpoints",
    f"{BASE_PATH}/models/loras",
    f"{BASE_PATH}/models/vae",
    f"{BASE_PATH}/models/controlnet",
    f"{BASE_PATH}/models/clip",
    f"{BASE_PATH}/models/unet",
    f"{BASE_PATH}/custom_nodes",
    f"{BASE_PATH}/output"
]

for d in directories:
    os.makedirs(d, exist_ok=True)
print("✅ Estructura de carpetas verificada en Google Drive.")

In [ ]:
# Paso 3: Instalación de ComfyUI y ComfyUI-Manager
import os
import subprocess

BASE_PATH = '/content/drive/MyDrive/ComfyUI'

if not os.path.exists(os.path.join(BASE_PATH, 'main.py')):
    print('Clonando repositorio de ComfyUI en Google Drive...')
    os.makedirs(BASE_PATH, exist_ok=True)
    clone = subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI', BASE_PATH], capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(f'Error al clonar ComfyUI: {clone.stderr}')
else:
    print('✅ ComfyUI ya está instalado en Google Drive.')

subprocess.run(['bash', '-lc', f'cd "{BASE_PATH}" && pip install -r requirements.txt --quiet'], check=False)

# Instalar ComfyUI-Manager
MANAGER_PATH = os.path.join(BASE_PATH, 'custom_nodes', 'comfyui-manager')
if not os.path.exists(MANAGER_PATH):
    print('Instalando ComfyUI-Manager...')
    subprocess.run(['git', 'clone', 'https://github.com/ltdrdata/ComfyUI-Manager', MANAGER_PATH], check=False)
else:
    print('✅ ComfyUI-Manager ya está instalado.')

In [ ]:
# Paso 4: Gestor interactivo de descargas (Formulario para Móvil / Android)
import os
import subprocess
import ipywidgets as widgets
from IPython.display import display

# Recomendación: usar variables de entorno para tokens (HF_TOKEN y CIVITAI_TOKEN).
HF_TOKEN = os.environ.get('HF_TOKEN')
CIVITAI_TOKEN = os.environ.get('CIVITAI_TOKEN')

subprocess.run(['apt-get', 'install', '-y', 'aria2', '--quiet'], check=False)

provider_dropdown = widgets.Dropdown(options=['Hugging Face', 'Civitai', 'URL Directa'], value='Hugging Face', description='Proveedor:')
resource_type = widgets.Dropdown(options=['checkpoints', 'loras', 'vae', 'controlnet', 'unet'], value='checkpoints', description='Tipo:')
url_input = widgets.Text(placeholder='Pegar enlace de descarga aquí', description='URL:', layout=widgets.Layout(width='90%'))
filename_input = widgets.Text(placeholder='Ej: modelo.safetensors', description='Nombre:', layout=widgets.Layout(width='90%'))
download_button = widgets.Button(description='Iniciar Descarga', button_style='primary', icon='download')
output_area = widgets.Output()

def download_with_aria2(url, target_folder, name, hf_token=None, civitai_token=None):
    os.makedirs(target_folder, exist_ok=True)
    command = [
        'aria2c',
        '--console-log-level=error',
        '-c', '-x', '16', '-s', '16', '-k', '1M',
    ]
    if hf_token:
        command += ['--header', f'Authorization: Bearer {hf_token}']
    command += ['-d', target_folder, '-o', name, url]
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return result.returncode, result.stdout

def run_download(b):
    with output_area:
        output_area.clear_output()
        url = url_input.value.strip()
        name = filename_input.value.strip()
        target_folder = f"/content/drive/MyDrive/ComfyUI/models/{resource_type.value}"
        os.makedirs(target_folder, exist_ok=True)

        if not url or not name:
            print('❌ Por favor introduce la URL y el nombre del archivo.')
            return

        target_file = os.path.join(target_folder, name)
        if os.path.exists(target_file):
            print(f'⚠️ El archivo {name} ya existe en {target_folder}. Skipping...')
            return

        if provider_dropdown.value == 'Civitai' and CIVITAI_TOKEN:
            url = f"{url}?token={CIVITAI_TOKEN}"

        print(f'
Descargando {name} en {target_folder}...')
        code, log = download_with_aria2(url, target_folder, name, hf_token=HF_TOKEN if provider_dropdown.value == 'Hugging Face' else None)
        if code == 0 and os.path.exists(target_file):
            print(f'✅ Descarga completada: {name}')
        else:
            print('❌ Error durante la descarga.')
            if log:
                print(log)

download_button.on_click(run_download)
display(widgets.VBox([provider_dropdown, resource_type, url_input, filename_input, download_button, output_area]))

In [ ]:
# Paso 5: Iniciar ComfyUI con Túnel de Cloudflare
import os
import re
import subprocess
import time

BASE_PATH = '/content/drive/MyDrive/ComfyUI'
os.chdir(BASE_PATH)

# Instalar Cloudflare Tunnel
cloudflared_bin = 'cloudflared-linux-amd64.deb'
subprocess.run(['wget', f'https://github.com/cloudflare/cloudflared/releases/latest/download/{cloudflared_bin}', '--quiet', '-O', cloudflared_bin], check=False)
subprocess.run(['dpkg', '-i', cloudflared_bin], check=False)
subprocess.run(['apt-get', '-f', 'install', '-y'], check=False)

print('🚀 Iniciando servidor ComfyUI y generando URL pública...')
cf_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
start = time.time()
while time.time() - start < 30:
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    match = re.search(r'https://[a-z0-9.-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('URL pública:', public_url or 'No encontrada. Revisa la salida del tunnel.')

# Inicia ComfyUI en segundo plano
main_proc = subprocess.Popen(['python', 'main.py', '--dont-print-server'], cwd=BASE_PATH)
print('✅ ComfyUI arrancado en segundo plano.')